# kernel

> The code-execution engine: a single shared IPython shell, plus the primitives that run a code cell's source and turn its stdout/stderr/rich-display output into an ordered list of output blocks -- both synchronously (`run_code`) and streamed from a background thread (`_run_code_bg` + `_RunState`/`_TeeStream`). Deliberately stateless about the notebook: no `nb`, no `_run_state` global, no rendering. `cells.py` owns those and drives these primitives.

In [ ]:
#| default_exp kernel

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import sys, base64, threading, io as _pyio
from toolslm.shell import get_shell
from IPython.utils.capture import capture_output

In [ ]:
#| export
_shell = get_shell()
_shell.system = _shell.system_piped   # capture `!cmd` output into stdout

# get_shell() builds a standalone TerminalInteractiveShell without registering it as IPython's
# active instance -- so capture_output() (which looks up get_ipython() internally to find a shell
# to capture display() calls from) can't find it and silently skips rich-output capture entirely.
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell._instance = _shell

try:
    import matplotlib
    matplotlib.use('Agg')  # headless -- figures are captured explicitly in run_code(), not via any interactive/inline backend
except ImportError:
    pass

# IPython disables every rich MIME formatter but text/plain on a bare shell -- turn the ones we
# know how to render back on, so e.g. a DataFrame's _repr_html_ or a returned PIL Image gets used
# (both for display(...) calls and for the auto-formatted last expression; see run_code()).
_MIME_PRIORITY = ('text/html', 'image/svg+xml', 'image/png', 'text/markdown', 'application/json')
for _mime in _MIME_PRIORITY:
    if _mime in _shell.display_formatter.formatters:
        _shell.display_formatter.formatters[_mime].enabled = True

In [ ]:
#| export
def _collapse_cr(s:str) -> str:
    "Collapse \\r-overwritten text (tqdm-style progress bars) down to each line's final state."
    return '\n'.join(ln.split('\r')[-1] for ln in s.split('\n'))

In [ ]:
#| export
def _best_block(fmt:dict) -> dict:
    "The richest available rendering of a formatted-object MIME dict, as an output block; falls back to its plain-text repr."
    for mime in _MIME_PRIORITY:
        if mime in fmt:
            return {'type':'display', 'mime':mime, 'data':fmt[mime]}
    return {'type':'stream', 'mime':None, 'data':fmt.get('text/plain', '')}

In [ ]:
#| export
def _flush_figures() -> list[dict]:
    "Any matplotlib figures left open after a cell runs, as image/png display blocks -- captured explicitly (savefig + close) rather than relying on an inline backend's event hooks, which proved unreliable on this headless shell."
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        return []
    blocks = []
    for num in plt.get_fignums():
        fig = plt.figure(num)
        buf = _pyio.BytesIO()
        fig.savefig(buf, format='png', bbox_inches='tight')
        blocks.append({'type':'display', 'mime':'image/png', 'data':base64.b64encode(buf.getvalue()).decode()})
        plt.close(fig)
    return blocks

In [ ]:
#| export
def run_code(src:str) -> list[dict]:
    "Execute `src` in the shared shell; return an ordered list of output blocks (each a dict with 'type' -- stream/error/display -- and, for display blocks, a 'mime' type). See Cell.output."
    with capture_output() as io:
        res = _shell.orig_run(src)  # bypass toolslm's run_cell wrapper -- it discards stderr and any display() output, keeping only stdout
    blocks = []
    if res.error_in_exec is not None:
        e = res.error_in_exec
        blocks.append({'type':'error', 'mime':None, 'data':f'{type(e).__name__}: {e}'})
    text = _collapse_cr((io.stdout or '') + (io.stderr or ''))
    if text:
        blocks.append({'type':'stream', 'mime':None, 'data':text})
    blocks += [_best_block(o.data) for o in io.outputs]  # explicit display(...) calls, in call order
    if res.result is not None:
        fmt, _md = _shell.display_formatter.format(res.result)
        blocks.append(_best_block(fmt))
    blocks += _flush_figures()
    return blocks

In [ ]:
#| export
class _RunState:
    "Tracks one in-flight background code-cell execution. Only one cell can run at a time (like a real kernel), so a single `_run_state` global is enough."
    def __init__(self, cell_id:int):
        self.cell_id = cell_id
        self.buffer:list[str] = []   # text chunks written so far, in arrival order
        self.done = False
        self.blocks:list[dict]|None = None
        self.thread:threading.Thread|None = None

In [ ]:
#| export
class _TeeStream(_pyio.TextIOBase):
    "A writable stream that appends every write() straight into a _RunState's buffer, so a poll request mid-execution can see output as it's produced -- unlike capture_output(), which only exposes text once its `with` block exits. Subclassing TextIOBase (rather than a bare object) gives it a real isatty()/readable()/etc. file protocol -- without it, libraries like tqdm that probe for a proper file object fall back to appending a newline per update instead of overwriting in place with '\\r'."
    def __init__(self, state:_RunState): self.state = state
    def writable(self) -> bool: return True
    def write(self, s:str) -> int:
        if s: self.state.buffer.append(s)
        return len(s)

In [ ]:
#| export
def _run_code_bg(src:str, state:_RunState) -> None:
    "Runs in a background thread: executes `src` with stdout/stderr tee'd into `state.buffer` as it goes, then fills in `state.blocks` (same shape as run_code()'s return) once execution finishes. See run_code_poll() for the other end."
    old_out, old_err = sys.stdout, sys.stderr
    sys.stdout = sys.stderr = _TeeStream(state)  # one shared stream, so stdout/stderr interleave in real chronological order
    try:
        with capture_output(stdout=False, stderr=False) as io:  # display()/last-expr capture only -- stdout/stderr are already tee'd above
            res = _shell.orig_run(src)
    finally:
        sys.stdout, sys.stderr = old_out, old_err
    blocks = []
    if res.error_in_exec is not None:
        e = res.error_in_exec
        blocks.append({'type':'error', 'mime':None, 'data':f'{type(e).__name__}: {e}'})
    text = _collapse_cr(''.join(state.buffer))
    if text:
        blocks.append({'type':'stream', 'mime':None, 'data':text})
    blocks += [_best_block(o.data) for o in io.outputs]
    if res.result is not None:
        fmt, _md = _shell.display_formatter.format(res.result)
        blocks.append(_best_block(fmt))
    blocks += _flush_figures()
    state.blocks = blocks
    state.done = True

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()